# Demo : AI Observability with TruLens and OpenTelemetry 


<span style="background-color:yellow; color:red">
## Before you start: Restart the kernel (Kernel → Restart) to clear any leftover variables from previous runs.
</span>

📚 **This is an OPTIONAL notebook. We will NOT dedicate class time for this lab. We recommend you try this lab outside of the dedicated class time. If you happen to have free extra time leftover during the class, please try this out as you wish!**

In this notebook you will learn and practice the following:

❄️ Setting up AI observability for RAG applications using TruLens

❄️ Implementing OpenTelemetry tracing for AI model interactions

❄️ Building instrumented RAG systems with Cortex Search and LLM functions

❄️ Running batch experiments and evaluating AI application performance

❄️ Computing and analyzing key metrics like answer relevance, context relevance, and groundedness



📌 **Note**: 

We acknowledge not everyone has **Python skills**. During the class, your instructor **will walk you through** these instructions and discuss how one would approach **observability** in the real world. 

In the **longer term**, many of these **observability** capabilities will be available **out‑of‑the‑box** in Snowsight.

Due to **regional workload spikes**, there may be **latency** with some of the steps.  
In the real world, for consistent performance, customers can explore **[Provisioned Throughput]** (https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

Version: 160904

## Introduction

In this lab, we will explore how to implement AI observability for Retrieval-Augmented Generation (RAG) applications using TruLens and OpenTelemetry tracing. As AI applications become more complex and critical to business operations, it's essential to have comprehensive monitoring, evaluation, and debugging capabilities.


**TruLens** is a leading open-source tool for evaluating and tracking AI applications, while **OpenTelemetry** provides industry-standard observability instrumentation. Together, they enable comprehensive monitoring of your AI systems running on Snowflake. For more details see https://www.trulens.org/?_fsi=MdHxDfd4

- 🔗 **Snowflake Docs (AI Observability with TruLens)**: [Evaluate AI applications with TruLens](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/evaluate-ai-applications)  
- 🔗 **TruLens Project Docs**: [TruLens Documentation](https://www.trulens.org/getting_started/)

The first link is Snowflake's own documentation on how to instrument and evaluate AI apps inside Snowflake using the TruLens SDK. The second is the broader open‑source TruLens site, which covers quickstarts, integrations (LangChain, LlamaIndex, Cortex, etc.), and API references.  

----

**AI Observability** provides deep insights into how your AI applications behave in production, helping you understand:

* **Performance Metrics**: How well your RAG system retrieves relevant context and generates accurate answers
* **Tracing**: Detailed execution traces showing the flow from user query through retrieval to final response
* **Evaluation**: Automated assessment of key quality metrics like relevance, groundedness, and coherence
* **Debugging**: Identification of bottlenecks, errors, and areas for improvement in your AI pipeline


In this lab, we'll build an instrumented RAG application that:
1. Uses Snowflake Cortex Search for document retrieval
2. Leverages Cortex LLM functions for answer generation  
3. Includes OpenTelemetry tracing for full observability
4. Runs the application with a preset batch of questions
5. Evaluates the application run for quality and performance
6. Stores results in Snowflake for analysis and visualization

## Getting Started 

In real-world AI applications, organizations need to ensure their systems are performing reliably and delivering high-quality results. This lab simulates a common scenario where we need to implement comprehensive observability for a production RAG system. To build effective AI observability, we'll follow these key steps:

* **Understand the AI application requirements**: Define what constitutes good performance for our RAG system in terms of accuracy, relevance, and user satisfaction

* **Implement proper instrumentation**: Add tracing and monitoring capabilities to capture detailed execution information without impacting performance

* **Select appropriate evaluation metrics**: Choose metrics that align with business objectives, such as answer relevance, context relevance, and groundedness

* **Design comprehensive test datasets**: Create or identify datasets that represent real user queries and expected responses

* **Set up automated evaluation pipelines**: Build systems that can continuously assess AI performance and detect degradation

* **Establish monitoring and alerting**: Create dashboards and alerts to track system health and performance trends

* **Enable debugging and root cause analysis**: Provide tools to investigate issues and understand system behavior at a granular level

In this lab, we're implementing observability for the **Travelbug** RAG application, which helps users get answers about travel activities and bookings. The business users need confidence that the AI system is providing accurate, relevant information to customers, and the technical team needs visibility into system performance and potential issues.

### Set up your current role, database, schema and warehouse

Before we begin implementing AI observability, we need to establish our Snowflake session context. This step configures the environment where our RAG application and observability data will be stored.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_GENAI_DB'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r dataframe_1
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA resources;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: AI Observability';
SHOW PARAMETERS LIKE 'query_tag' IN SESSION ->>
  SELECT "value" AS query_tag FROM $1;

### Install the dependecies once per kernel session

### Install Trulens packages.

The following packages are required for this notebook and are installed automatically by the pip cell above:

  - trulens-core
  - trulens-connectors-snowflake
  - trulens-providers-cortex
  - trulens-feedback
  - snowflake-sqlalchemy

In most cases, the pip cell handles everything and no manual action is needed.

If, however, you need to reinstall them (for example, if something was removed or didn't install correctly), restart the kernel and re-run the pip cell below. It will detect that `trulens-core` is missing and reinstall all packages.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess, importlib, sys

try:
    v = version('trulens-core')
    print(f"TruLens {v} already installed — skipping pip.")
except PackageNotFoundError:
    print("Installing TruLens...")
    subprocess.check_call(
        ['pip', 'install', '-q', 'trulens-core==2.13.1', 'trulens-connectors-snowflake',
         'trulens-feedback', 'trulens-providers-cortex', 'snowflake-sqlalchemy',
         'sqlalchemy>=2.0,<2.1']
    )
    importlib.invalidate_caches()
    for mod_name in list(sys.modules.keys()):
        if mod_name.startswith('snowflake.sqlalchemy') or mod_name.startswith('trulens'):
            del sys.modules[mod_name]
    v = version('trulens-core')
    print(f"Packages installed successfully. TruLens {v}")

### Database privileges required for AI observability.

In order for you to complete this lab, we have already granted the following **privileges** to the **GENAI_ROLE**.

*   **SNOWFLAKE.CORTEX_USER** database role
*   **SNOWFLAKE.AI_OBSERVABILITY_EVENTS_LOOKUP** application role
*   **CREATE EXTERNAL AGENT** privilege on the **RESOURCES** schema


📌 **Note**: 

When you create a Cortex Search Service, the initial search index build usually completes in seconds but can sometimes take a few minutes. This variability is due to the embedding step, which calls a large language model (LLM) in the region. A higher workload in the region will result in slightly longer build times.

## Set up Cortex Search Services

The next cell contains the SQL commands to create a Cortex Search service using the PDF documents stored in our stage named **search_docs**. These commands are safe to run multiple times and are designed to initialize the search service as needed whether you're starting fresh or continuing from a previous setup. (You may have performed these commands previously in a different exercise--running them again will not cause any issues.)

Key steps:

* **Create a table** to store extracted text and metadata from PDF documents stored in a Snowflake stage.
* **Extract text** from each document using an AI parsing function and load the results into the table.
* **Chunk the text** into smaller, overlapping segments to prepare it for classification and search.
* **Classify each chunk** into predefined categories using a machine learning model.
* **Create a Cortex Search Service** that indexes the chunked and categorized text, enabling natural language search across the documents.

In [ ]:
%%sql -r dataframe_2
-- Create and populate PARSE_DOC in one step
CREATE TABLE IF NOT EXISTS {{user}}_genai_db.raw.PARSE_DOC AS
SELECT
    relative_path AS relative_path,
    GET_PRESIGNED_URL('@{{user}}_genai_db.resources.genai2day', relative_path) AS source_url,
    size AS size,
    TO_VARCHAR(
        AI_PARSE_DOCUMENT(
            TO_FILE('@{{user}}_genai_db.resources.genai2day', relative_path),
            {'mode': 'OCR'}
        ):content
    ) AS raw_text
FROM DIRECTORY(@{{user}}_genai_db.resources.genai2day)
WHERE STARTSWITH(relative_path, 'search_docs/');

In [ ]:
%%sql -r dataframe_3
-- Create table with explicit column list
CREATE or replace TABLE {{user}}_genai_db.raw.PARSE_DOC_CHUNKS (
    doc_id,
    relative_path,
    category,
    chunk,
    source_url
) AS
SELECT
    UUID_STRING() AS doc_id,
    relative_path,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        TO_VARCHAR(c.value),
        ['sunsettour', 'mountainhike', 'winetour', 'balloonride', 'snorkeling']
    ):label::VARCHAR AS category,
    TO_VARCHAR(c.value) AS chunk,
    source_url
FROM {{user}}_genai_db.raw.PARSE_DOC,
     LATERAL FLATTEN(
         input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
             raw_text,
             'markdown',
             2000,
             500
         )
     ) c;

In [ ]:
%%sql -r dataframe_4
-- Create a Cortex Search Service
CREATE or replace CORTEX SEARCH SERVICE {{user}}_genai_db.resources.travelbug_search
ON chunk
ATTRIBUTES category
WAREHOUSE = {{user}}_genai_wh
TARGET_LAG = '1 minute'
EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT
        doc_id,
        chunk,
        relative_path,
        source_url,
        category::VARCHAR AS category
    FROM {{user}}_genai_db.raw.parse_doc_chunks
);

## Building The Retrieval Component : Connecting Cortex Search

The first component of our observable RAG system is the **retrieval mechanism**. 

We'll use Snowflake's **Cortex Search Service**, which provides high-performance semantic search capabilities powered by state-of-the-art embedding models as the basis of this lab to demonstrate AI observability capability.

For our Travelbug application, we have built the search service called **TRAVELBUG_SEARCH** that indexes travel-related documents, activity descriptions, and booking information.

### CortexSearchRetriever Class

The following implementation creates a retrieval wrapper that does the following: 

* Connects to our pre-configured Cortex Search service
* Accepts natural language queries from users
* Returns the most relevant document chunks for context
* Limits results to avoid overwhelming the LLM with too much context

📌 **Note**: 

The next cell does not produce any output.

<span style="background-color:yellow; color:red">Run it once per kernel session </span>. The class is defined in memory and will persist until the kernel restarts


In [ ]:
from snowflake.core import Root
from typing import List
from snowflake.snowpark.session import Session

class CortexSearchRetriever:

    def __init__(self, snowpark_session: Session, limit_to_retrieve: int = 4):
        self._snowpark_session = snowpark_session
        self._limit_to_retrieve = limit_to_retrieve

    def retrieve(self, query: str) -> List[str]:
        root = Root(self._snowpark_session)
        current_db = self._snowpark_session.get_current_database().strip('"')
        search_service = (root
            .databases[current_db]
            .schemas["RESOURCES"]
            .cortex_search_services["TRAVELBUG_SEARCH"]
        )
        resp = search_service.search(
            query=query,
            columns=["chunk"],
            limit=self._limit_to_retrieve
        )
        if resp.results:
            return [curr["chunk"] for curr in resp.results]
        else:
            return []

## Enabling OpenTelemetry Tracing for AI Observability

Before building our RAG system, we need to enable comprehensive tracing capabilities. https://opentelemetry.io/ is an industry-standard observability framework that provides distributed tracing, metrics, and logging for complex applications.

### Why OpenTelemetry for AI Applications?

OpenTelemetry tracing is particularly valuable for AI systems because:

* **Distributed Operations**: RAG systems involve multiple components (retrieval, LLM calls, post-processing) that need coordinated monitoring
* **Performance Analysis**: Track latency and throughput across each component to identify bottlenecks  
* **Error Debugging**: Capture detailed context when things go wrong to enable faster root cause analysis
* **Quality Correlation**: Link performance metrics with quality metrics to understand trade-offs
* **Production Monitoring**: Provide real-time visibility into system health and user experience

### TruLens Integration

**TruLens** integrates seamlessly with **OpenTelemetry** to provide AI-specific observability:
* Automatic instrumentation of LLM calls and RAG operations
* Custom span attributes for AI-specific metrics (relevance, groundedness, etc.)
* Integration with evaluation frameworks for quality assessment
* Storage and visualization of traces alongside performance data

### Enable tracing.

We will enable tracing using the **TRULENS_OTEL_TRACING** parameter.

* **TRULENS_OTEL_TRACING=1**: This environment variable tells TruLens to emit OpenTelemetry traces
* **Automatic Instrumentation**: TruLens will automatically capture trace data for all instrumented operations
* **Zero Performance Impact**: OpenTelemetry is designed for production use with minimal overhead
* **Distributed Context**: Traces will include full context propagation across service boundaries
* **Integration Ready**: Traces can be exported to any OpenTelemetry-compatible backend (Jaeger, Zipkin, Cloud providers, etc.)


📌 **Note**: 

The line **os.environ["TRULENS_OTEL_TRACING"] = "1"** enables OpenTelemetry tracing for TruLens by setting an environment variable inside your Snowflake Notebook's Python session. This ensures that any functions you instrument with TruLens will automatically generate trace spans for observability.



<span style="background-color:yellow; color:red">Run it once per kernel session </span> , sets an environment variable in memory that persists until the kernel restarts.

In [ ]:
import os
os.environ["TRULENS_OTEL_TRACING"] = "1"

## Building an Instrumented RAG System--with Instrumentation

Now we'll create a complete RAG (Retrieval-Augmented Generation) system with comprehensive instrumentation. This implementation demonstrates how to add observability to each component of the RAG pipeline without affecting the core functionality.

### Understanding RAG Architecture

A typical RAG system has three main components:
1. **Retrieval**: Find relevant documents/context for the user's question
2. **Generation**: Use an LLM to generate an answer based on the retrieved context  
3. **Orchestration**: Coordinate the flow and handle the overall user interaction

Below is a simple explanation of what the code in the next cell is designed to do.

**Creates a "Look-It-Up-First" AI**: 
This code builds a smart system that doesn't just answer from memory. It first researches the topic and then forms an answer.

**Two-Step** Process:

*   **Find (Retrieve)**: 
When you ask a question, it first searches a private database to find relevant documents or text snippets. Think of it as a super-fast librarian finding the exact pages you need.

*   **Answer (Generate)**: 
It then gives your question and the information it found to a powerful AI (**claude-haiku-4-5**). It instructs the AI to create a detailed answer using only the provided information.

**Prevents Guesswork**: 
This method ensures the AI's answers are based on specific, factual data, which greatly reduces the chance of it making things up (hallucinating).

**Includes a Monitor**: 
The code uses **TruLens** to watch and track how well the system is finding information and generating answers. This helps developers check its accuracy and make improvements.

 <span style="background-color:yellow; color:red">Run it once per kernel session </span>

In [ ]:
import sys
from snowflake.cortex import complete
from trulens.apps.app import TruApp
from trulens.connectors.snowflake import SnowflakeConnector
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

# Fix for Snowflake notebooks: __main__ module needs __package__ attribute
if not hasattr(sys.modules['__main__'], '__package__'):
    sys.modules['__main__'].__package__ = None

class RAG:

    def __init__(self):
        self.retriever = CortexSearchRetriever(snowpark_session=session, limit_to_retrieve=4)

    # Monitor the search step - captures the query and the retrieved chunks
    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
        }
    )
    def retrieve_context(self, query: str) -> list:
        """Retrieve relevant text from vector store."""
        return self.retriever.retrieve(query)

    # Monitor the LLM step — captures the generated response
    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def generate_completion(self, query: str, context_str: list) -> str:
        """Generate answer from context."""
        prompt = f"""You are an expert assistant extracting information from the provided context.
Answer the question fully, in long form, using only the context.
If the context does not contain the answer, say so.

Context: {context_str}
Question: {query}
Answer:"""
        response = complete("claude-haiku-4-5", prompt)
        return response

    # Monitor the full pipeline — ties retrieval and generation into one trace

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        })
    def query(self, query: str) -> str:
        context_str = self.retrieve_context(query)
        return self.generate_completion(query, context_str)

rag = RAG()

## Registering the RAG Application with TruLens

Now we need to **register** our instrumented RAG application with TruLens for tracking and evaluation. This step creates a managed application instance that can run experiments, store results, and provide observability data.


### Key Configuration Parameters

The code below supplies the following parameters to register the application.

* **app_name**: Unique identifier for your RAG application (**{{user}}_travel_rag**)
* **app_version**: Version identifier to track different implementations
* **connector**: SnowflakeConnector links TruLens to your Snowflake session for data storage
* **Application Instance**: The actual RAG object that will be instrumented and evaluated

This registration creates a bridge between your application code and the TruLens evaluation framework, enabling comprehensive AI observability.

📌 Note:

If you receive any **warning** message while running the cells, you can safely **ignore** it and continue with the next step.

### Registering the RAG Application with TruLens

 <span style="background-color:yellow; color:red">Run it once per kernel session </span>

In [ ]:
from trulens.connectors.snowflake import SnowflakeConnector

tru_snowflake_connector = SnowflakeConnector(snowpark_session=session)

app_name = f"{user}_travel_rag"
app_version = "cortex_search"

tru_rag = TruApp(
    rag,
    app_name=app_name,
    app_version=app_version,
    connector=tru_snowflake_connector
)

## Configuring and Running Evaluation Experiments

To properly evaluate our RAG system, we need to run systematic experiments using representative test datasets. This section shows how to configure batch evaluations that will assess the quality and performance of our AI application.

### Understanding Evaluation Datasets

For effective AI observability, we need structured test datasets that include:

* **Query Examples**: Representative user questions that mirror real-world usage
* **Ground Truth Responses**: Expected or ideal answers for quality comparison
* **Diverse Scenarios**: Different types of questions, complexity levels, and edge cases
* **Business Context**: Questions that align with actual business use cases

### Dataset Configuration Options

TruLens supports multiple dataset sources:

1. **Python DataFrames**: For programmatically generated or manipulated test sets
2. **Snowflake Tables**: For larger datasets stored directly in your data warehouse (our approach)
3. **External Files**: CSV, JSON, or other structured formats

### Setting Up the Test Dataset

For this lab, we're using a pre-configured dataset stored in Snowflake:

* **Dataset Name**: **SEARCH_OBSERVABILITY_DATA** - Contains travel-related queries and expected responses
* **Input Column**: **QUERY** - User questions about travel activities, bookings, and services  
* **Ground Truth Column**: **GROUND_TRUTH_RESPONSE** - Verified correct answers for comparison
* **Purpose**: Enables automated evaluation of answer quality, relevance, and accuracy

### Experiment Configuration Parameters

* **run_name**: Unique identifier for this specific experiment run
* **dataset_name**: References the Snowflake table containing test queries
* **description**: Human-readable explanation of what this experiment tests
* **label**: Categorization tag for organizing related experiments
* **source_type**: Specifies "TABLE" for Snowflake-based datasets

This configuration enables TruLens to automatically run your RAG application against the entire test dataset and capture comprehensive performance metrics.

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TABLE {{user}}_GENAI_DB.RESOURCES.SEARCH_OBSERVABILITY_DATA (
    ID INT AUTOINCREMENT,
    QUERY STRING,
    GROUND_TRUTH_RESPONSE STRING);


INSERT INTO {{user}}_GENAI_DB.RESOURCES.SEARCH_OBSERVABILITY_DATA (query, ground_truth_response)
VALUES
    ('Where does Wine Tour run from?', 'Wine tour is run from Vineyard Valley'),
    ('What is the third stop of the wine tour?', 'The third stop of the wine tour is Coastal Vineyard'),
    ('How much does it cost for transportation for the Wine Tour?','Shuttle available for pickup and drop from downtown which costs $30.00.'  );
    

### Tracking Runs with TruLens

To evaluate and observe your application with TruLens, you first define a **RunConfig**.  
This configuration specifies the unique name of the run, the dataset to use, and descriptive metadata such as a human‑readable description and a label tied to the current user. The **source_type** indicates where the data comes from (in this case, a table), and the **dataset_spec** maps which column contains the input queries and which column contains the ground truth responses.  

Once the configuration is defined, you create a **Run** by passing the **RunConfig** into **tru_rag.add_run**. This registers the run with TruLens so that evaluations, traces, and results can be logged and later analyzed.  

In the example, the run is named **"experiment_1_run"**, uses the dataset **SEARCH_OBSERVABILITY_DATA**, and is described as 'Questions about the travelbug tours' The label incorporates the **user** variable so each participant's run is uniquely identified. The dataset specification maps the **QUERY** column as the input and the **GROUND_TRUTH_RESPONSE** column as the expected output.

### Configure and create an evaluation run



In [ ]:
# Available LLM judges (copy one into llm_judge_name):

# Recommended
# llama3.3-70b        — 0.79/1.00  — Strong reasoning, reliable, cost-effective, higher accuracy
# claude-sonnet-4-5   — 3.00/15.00 — Highest quality structured output
# openai-gpt-5-mini   — 0.30/1.20  — Best value, cheap with high accuracy

# Usable
# llama3.1-70b        — 0.79/1.00  — Older gen, slightly less consistent (default)
# mistral-large2      — 2.00/6.00  — Decent but pricey for moderate accuracy
# gemini-3.1-pro      — 1.25/5.00  — Strong model, Deep reasoning, Strong instruction-following

# Not recommended
# mistral-large3      — 2.00/6.00  — Fails on 3/4 metrics, unreliable
# llama3.1-8b         — 0.19/0.19  — Too small for eval reasoning
# claude-haiku-4-5    — 0.25/1.25  — Too shallow for judging


from trulens.core.run import Run, RunConfig
from datetime import datetime, timezone

run_name = f"experiment_{datetime.now():%H-%M-%S}"

run_config = RunConfig(
    run_name=run_name,
    dataset_name=f"{user}_GENAI_DB.RESOURCES.SEARCH_OBSERVABILITY_DATA",
    description="Questions about the travelbug tours",
    label=f"{user}_travel_eval",
    source_type="TABLE",
    dataset_spec={
        "input": "QUERY",
        "ground_truth_output": "GROUND_TRUTH_RESPONSE",
    },
    llm_judge_name="llama3.3-70b"
)

run: Run = tru_rag.add_run(run_config=run_config)
print(f"Run created: {run_name}")

## Executing the Batch Evaluation Experiment

Now we're ready to execute our comprehensive evaluation experiment. This process will run our instrumented RAG application against the entire test dataset, capturing detailed performance and quality metrics for each query.

### Understanding Batch Execution

When we start the experiment run, TruLens will:

1. **Load the Dataset**: Retrieve all test queries from the specified Snowflake table
2. **Process Each Query**: Run your RAG application for every question in the dataset
3. **Capture Traces**: Record detailed OpenTelemetry traces for each execution
4. **Store Results**: Save all outputs, intermediate steps, and metadata to Snowflake
5. **Prepare for Evaluation**: Structure data for automated quality assessment

### What Gets Captured

During execution, the system captures:
* **Input Queries**: Original user questions from the test dataset
* **Retrieved Context**: Documents/chunks returned by Cortex Search
* **LLM Responses**: Generated answers from the language model
* **Execution Traces**: Detailed timing and performance data
* **Error Information**: Any failures or issues encountered

### Resource Management

In a production setup, you may need to scale up your warehouse size to accommodate the load. In the lab environment, we can use the existing warehouse.


### Running the Evaluation with TruLens

Once you have defined and registered a run with **RunConfig**, the next step is to **start the run**.  
This is done with the command:

When you call **run.start()**, TruLens begins executing the evaluation you configured.  
It takes the dataset you specified (for example, queries and their ground‑truth responses), feeds the inputs into your application, captures the outputs, and records the results.  

During this process, TruLens also logs traces and evaluation metrics so you can later analyze how well your application performed.  
In short, **run.start()** is what actually **launches the experiment** and ensures the results are tracked for observability. The next cell may take 30 seconds or so to complete.

### Launch and monitor the evaluation run.

In [ ]:
run.start()

#### Monitoring batch run submission
The method call **run.start()** above submits a batch run of your RAG system, asking it the provided questions, getting responses, and collecting telemetry on the run as instrumented in your code. This batch run takes place asynchronously to your notebook, so you need to monitor its progress before proceeding. 

Below, call **run.get_status()** to monitor the state of your experiment. The code checks repeatedly until you get status **RunStatus.INVOCATION_COMPLETED** before continuing. (A status of **RunStatus.COMPLETED** will appear if you have already run the rest of this notebook.)

In [ ]:
from trulens.core.run import RunStatus
from time import sleep
from datetime import datetime

print(f"Batch run for experiment: {run_name}")
print("------------------------")

while True:
    status = RunStatus(run.get_status())
    print(f"{datetime.now():%H:%M:%S} {status.value}")
    if status is RunStatus.INVOCATION_COMPLETED:
        break
    if status is RunStatus.INVOCATION_PARTIALLY_COMPLETED:
        print("WARNING: some records failed to invoke — metrics will be incomplete.")
        break
    if status in (RunStatus.FAILED, RunStatus.CANCELLED):
        raise RuntimeError(f"Invocation ended as {status.value}")
    sleep(5)

### What happens at the end of an execution run?

* **Batch Processing**: TruLens has processed all queries in your test dataset through the RAG application
* **Data Captured**: Complete traces, inputs, outputs, and metadata are now stored in Snowflake tables
* **Performance Data**: Timing information for retrieval, generation, and end-to-end processing has been collected
* **Quality Data**: Responses are ready for automated evaluation against ground truth answers

The evaluation run has captured comprehensive data about your RAG system's behavior, but quality metrics still need to be computed in the next step.

## Computing Quality and Performance Metrics

The final step in our AI observability pipeline is to compute comprehensive quality metrics for our RAG system. TruLens provides automated evaluation capabilities that assess key dimensions of AI application performance.

### Understanding AI Quality Metrics

The three core metrics we'll compute represent fundamental aspects of RAG system quality:

#### Answer Relevance
* **Purpose**: Measures how well the generated answer addresses the user's specific question
* **Methodology**: Uses LLM-based evaluation to assess relevance and completeness
* **Scale**: Typically 0-1, where 1 indicates perfect relevance
* **Business Impact**: Directly correlates with user satisfaction and task completion

#### Context Relevance
* **Purpose**: Evaluates whether the retrieved documents/chunks are relevant to the query
* **Methodology**: Analyzes semantic similarity between query and retrieved context
* **Scale**: 0-1, where 1 means all retrieved context is highly relevant
* **Business Impact**: Indicates retrieval system effectiveness and search quality

#### Groundedness
* **Purpose**: Assesses whether the generated answer is factually supported by the retrieved context
* **Methodology**: Checks for hallucinations and unsupported claims in the response
* **Scale**: 0-1, where 1 means the answer is fully grounded in provided context
* **Business Impact**: Critical for accuracy and trustworthiness in business applications

### Automated Evaluation Process

When computing metrics, TruLens:
1. **Loads Captured Data**: Retrieves all traces and outputs from the experiment run
2. **Applies Evaluation Models**: Uses specialized LLMs to assess each quality dimension
3. **Computes Scores**: Generates numerical scores for each metric on each query
4. **Aggregates Results**: Provides summary statistics and distributions
5. **Stores in Snowflake**: Saves all metric data for analysis and visualization

### Production Benefits

These metrics enable:
* **Performance Monitoring**: Track quality trends over time
* **Regression Detection**: Identify when changes degrade system performance  
* **Comparative Analysis**: Compare different model configurations or versions
* **Quality Assurance**: Set thresholds and alerts for production systems



### Computing Evaluation Metrics with TruLens

After you have run your evaluation, the next step is to compute metrics that measure the quality of the model's responses.  
At this stage, TruLens applies a set of evaluation criteria to the results of your run:

- **Answer relevance** — how well the model's answer matches the intent of the query.  
- **Context relevance** — whether the retrieved context was useful and appropriate for the question.  
- **Groundedness** — whether the answer is supported by the provided context rather than hallucinated.  
- **Correctness** — how closely the model's answer matches the ground truth response in your dataset.  

This step produces the key scores that help you understand how accurate, relevant, and reliable your application's outputs are.

### Gather metrics

In [ ]:
run.compute_metrics([
    "answer_relevance",
    "context_relevance",
    "groundedness",
    "correctness"
])

#### Monitor Progress of Experiment Evaluation

Like the batch run of the experiment at method call **run.start()**, the call to **run.compute_metrics()** evaluates the program results in a background, asynchronous process. The code below polls the status of this process until it completes. This is likely to take a minute or two.

In [ ]:
print(f"Evaluating results of experiment: {run_name}")
print("--------------------------------")

while True:
    status = RunStatus(run.get_status())
    print(f"{datetime.now():%H:%M:%S} {status.value}")
    if status is RunStatus.COMPLETED:
        break
    if status is RunStatus.PARTIALLY_COMPLETED:
        print("WARNING: some metrics did not compute. Check the Evaluations UI.")
        break
    if status in (RunStatus.FAILED, RunStatus.CANCELLED):
        raise RuntimeError(f"Metric computation ended as {status.value}")
    sleep(10)

### View Evaluation results

### "Results: Evaluation scorecard — all runs by llm judge (most recent first)"

In [ ]:
%%sql -r dataframe_8
-- Scoped to this app's External Agent object, so genai_role can read it with
-- USAGE on the agent plus CORTEX_USER. Direct SELECT on
-- SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS is reserved for admin use.

SELECT 
    RUN_NAME,
    LLM_JUDGE,
    TO_VARCHAR(ROUND(AVG(ar) / 3, 2), '0.00') AS "AR",
    TO_VARCHAR(ROUND(AVG(cr) / 3, 2), '0.00') AS "CR",
    TO_VARCHAR(ROUND(AVG(ac) / 3, 2), '0.00') AS "AC",
    TO_VARCHAR(ROUND(AVG(g) / 3, 2), '0.00') AS "G"
FROM (
    SELECT 
        RECORD_ATTRIBUTES:"snow.ai.observability.run.name"::STRING AS RUN_NAME,
        RECORD_ATTRIBUTES:"ai.observability.eval.llm_judge_name"::STRING AS LLM_JUDGE,
        TIMESTAMP,
        CASE WHEN RECORD_ATTRIBUTES:"ai.observability.eval.metric_name"::STRING = 'answer_relevance' THEN RECORD_ATTRIBUTES:"ai.observability.eval.score"::FLOAT END AS ar,
        CASE WHEN RECORD_ATTRIBUTES:"ai.observability.eval.metric_name"::STRING = 'context_relevance' THEN RECORD_ATTRIBUTES:"ai.observability.eval.score"::FLOAT END AS cr,
        CASE WHEN RECORD_ATTRIBUTES:"ai.observability.eval.metric_name"::STRING = 'correctness' THEN RECORD_ATTRIBUTES:"ai.observability.eval.score"::FLOAT END AS ac,
        CASE WHEN RECORD_ATTRIBUTES:"ai.observability.eval.metric_name"::STRING = 'groundedness' THEN RECORD_ATTRIBUTES:"ai.observability.eval.score"::FLOAT END AS g
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        '{{user}}_GENAI_DB', 'RESOURCES', '{{app_name}}', 'EXTERNAL AGENT'
    ))
    WHERE RECORD_TYPE = 'SPAN'
      AND UPPER(RECORD_ATTRIBUTES:"snow.ai.observability.object.name") = UPPER('{{app_name}}')
      AND RECORD_ATTRIBUTES:"ai.observability.eval.metric_name" IS NOT NULL
)
GROUP BY 1, 2
ORDER BY MAX(TIMESTAMP) DESC;

## Evaluation Results

To view evaluation results:

* Leave the current window running until the metric evaluations are completed.
* In the navigation pane on the left, hover over **AI & ML**. Right‑click on **Evaluations** and select the option to open it in a **new tab**.
* Select {{user}}_TRAVEL_RAG to view the runs and summary scores for the following

Your results may vary:

*  **Answer Relevance** (Avg):1
*  **Context Relevance** (Avg):0.28
*  **Correctness** (Avg):1
*  **Groundedness** (Avg):0.94

This is where it gets interesting! Why did the LLM generate such low scores for **Context Relevance**?


**Context Relevance** determines if the retrieved context from the **retriever** or the **search service** is **relevant** to the user query. Given the user query and retrieved context, an LLM judge is used to determine relevance of the retrieved context based on the query.

Required Attributes:

RETRIEVAL.QUERY_TEXT: User query in a RAG or search application

RETRIEVAL.RETRIEVED_CONTEXTS: Context retrieved from the search service or retriever


1. If you haven't already done so, click on the row for your evaluation run, then click on the line "What is the third stop of the wine?" It has the lowest Answer Relevance Score.


2. Expand on the **Context Relevance 0.17** panel to see a detailed explanation. This is an aggregated metric that measures how relevant the retrieved information is to the user's query. The score typically ranges from 0 to 1, where 1 signifies perfect relevance and 0 signifies no relevance. A score of 0.17 is on the lower end of this scale, suggesting that in most cases, the retrieved context is not a good match for the user's query. This can lead to inaccurate or irrelevant answers from your AI application.

### How To Investigate and improve the scores:

To understand the reason for the low score and improve the application's performance, here are some of the key considerations.

**Examine the Record details**: In the details, carefully **examine the specific user query** and the **context that was retrieved (Output)**. You will likely find a mismatch between the user's intent and the information your application is fetching.

**Review the LLM Judge's Explanation**: Expand the cell **Context Relevance** to read an explanation from the LLM judge on why it assigned that particular score. This explanation is crucial for understanding the specific issues with the retrieved context.

**Analyze Your Retrieval Process**: A low context relevance score often points to a problem with your retrieval-augmented generation (RAG) system. You may need to:

**Improve your embedding model**: The model you're using to create vector embeddings of your documents might not be capturing the semantic meaning of your content effectively.

**Refine your chunking strategy**: The way you are splitting your documents into smaller chunks for retrieval might be breaking up important context.

**Adjust your search parameters**: You may need to experiment with different settings for your vector search to improve the quality of the retrieved documents.

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What is the primary purpose of TruLens in AI observability?", "options": ["A) To generate AI responses faster", "B) TruLens provides evaluation metrics for RAG systems including answer relevance and groundedness", "C) To store conversation history", "D) To reduce model inference costs"], "hash": "cd21d356ea195f12e5f55a2cf8d9bc05"},
    {"q": "What does OpenTelemetry tracing provide for AI applications?", "options": ["A) It generates training data automatically", "B) It replaces the need for logging", "C) OpenTelemetry tracing enables detailed visibility into AI application execution flow", "D) It only monitors network latency"], "hash": "c5da9cfaea2f856f4db9932b87e5baa3"},
    {"q": "What does the groundedness metric measure in a RAG system?", "options": ["A) Groundedness measures how well the response is supported by the retrieved context", "B) The speed of response generation", "C) The number of tokens in the response", "D) User satisfaction with the answer"], "hash": "7a989e807f6e171017c327d4209b4913"},
    {"q": "Why is running evaluation experiments important for AI applications?", "options": ["A) It is only required for compliance purposes", "B) Evaluation is optional for production systems", "C) It only measures response time", "D) Running evaluation experiments helps identify quality issues before deploying to production"], "hash": "16dca8766ebdd22699fbbee72a320bc2"},
    {"q": "What does context relevance measure in RAG evaluation?", "options": ["A) The length of retrieved documents", "B) Context relevance measures how well the retrieved documents match the user query", "C) The number of documents retrieved", "D) The storage size of the context"], "hash": "c1d83887eb339c1c2e0c659778c8f390"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '✅' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))